### Bfactor Example

In [115]:
### importing into mdanalysis

import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
import numpy as np

In [116]:
def rmsfCal(u, outpath = None, sel = "protein and name CA", avg_str = None):

    average = align.AverageStructure(u, u, select=sel, ref_frame=0).run()
    ref = average.universe
    aligner = align.AlignTraj(u, ref, select=sel, in_memory=True).run() 
    c_alphas = u.select_atoms(sel)
    
    R = rms.RMSF(c_alphas).run()
    rmsf_results = R.results.rmsf
    print (f"CA: {len(c_alphas), len(rmsf_results)}")
    if outpath:
        u.trajectory[-1] # set the trajectory to the last frame, this will write the bfactor to the last frame
        u.add_TopologyAttr('tempfactors') # add empty attribute for all atoms
        protein = u.select_atoms(sel) # select protein atoms
        for residue, r_value in zip(protein.residues, rmsf_results):
            residue.atoms.tempfactors = r_value
        
        np.savetxt(outpath + 'rmsf.csv', rmsf_results, delimiter=',')
        u.atoms.write(outpath + 'bfactor_rmsf.pdb')
        #protein.write(Outfile[:-4] + '.xtc', frames='all')
        #protein.write(Outfile[:-4] + '.gro', reindex=False)

        avg_universe = mda.Merge(u.select_atoms("protein or resname THP1 or resname SP2 "))  # Create a new Universe for the sel
        avg_universe.add_TopologyAttr('tempfactors')
        for residue, r_value in zip(avg_universe.residues, rmsf_results):
            residue.atoms.tempfactors = r_value
        avg_universe.atoms.write(outpath + 'AvgProtStrBeta.pdb')
    return rmsf_results, u


In [117]:

path = "/mnt/mydrive/alpfaFold3/dynamics/Custom_MSA/dclk1_roundworm/dclk_middle_msa_custommsa/"
sel = "protein and name CA"
u = mda.Universe(path + 'frm0.pdb', path + 'traj.trr')
rmsf, u1 = rmsfCal(u, outpath = path , sel = sel)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(3.5, 2.2), dpi = 600, constrained_layout=True)
c_alphas = u.select_atoms(sel)
ax.plot(c_alphas.resids, rmsf, color = "black", label =path.split("/")[-2])
ax.legend(fontsize = 6, ncol=2);
from matplotlib.ticker import (MultipleLocator)
ax.xaxis.set_major_locator(MultipleLocator(50))
ax.xaxis.set_minor_locator(MultipleLocator(25))
#ax.yaxis.set_major_locator(MultipleLocator(5))
#ax.yaxis.set_minor_locator(MultipleLocator(2.5))
ax.tick_params(axis='both', which='major', labelsize=8)
ax.tick_params(axis='both', which='minor', labelsize=8)
ax.set_xlabel("Residue index", fontsize=10)
ax.set_ylabel("RMSF (\u00c5)", fontsize=10)
fig.savefig(path +  "RMSF.png", dpi = 600)



CA: (239, 239)


FileNotFoundError: [Errno 2] No such file or directory

In [ ]:
# import nglview as nv
# view = nv.show_mdanalysis(u.select_atoms('protein'))
# view.update_representation(color_scheme='bfactor')
# view
